# Pyomo: persistent solves and explicit checks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/pyomo-repeated-solves.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup

Colab already provides the general-purpose scientific libraries. This cell ensures the tested Pyomo and HiGHS versions; pip keeps an already-installed matching version. It does not replace Colab's NumPy, pandas or Matplotlib just to match the maintenance environment.


In [ ]:
# Use installed packages; install only missing ones, without version pins.
from importlib.util import find_spec
missing_packages = [name for name in ['pyomo', 'highspy'] if find_spec(name) is None]
if missing_packages:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])


In [ ]:
import pyomo.environ as pyo
from pyomo.opt import assert_optimal_termination
SOLVER = "appsi_highs"
assert pyo.SolverFactory(SOLVER).available(exception_flag=False)


## Reuse a model when the data change
This small example demonstrates a mutable parameter and the APPSI HiGHS interface. The solver interface detects relevant changes between solves. Reuse is useful for sensitivity experiments, but you must still check termination and validate the returned values after each change.

The collection targets Pyomo 6.10.1. It uses the documented APPSI interface rather than silently mixing the different result objects of the newer development-preview solver APIs. See [Pyomo's APPSI documentation](https://pyomo.readthedocs.io/en/stable/reference/topical/appsi/appsi.html).


In [ ]:
from pyomo.contrib.appsi.solvers import Highs
from pyomo.contrib.appsi.base import TerminationCondition
m = pyo.ConcreteModel('Resource experiment')
m.capacity = pyo.Param(initialize=10,mutable=True)
m.x = pyo.Var(domain=pyo.NonNegativeReals)
m.profit = pyo.Objective(expr=3*m.x,sense=pyo.maximize)
m.limit = pyo.Constraint(expr=2*m.x <= m.capacity)
solver = Highs()
observations = []
for capacity in [10,12,8]:
    m.capacity.set_value(capacity)
    result = solver.solve(m)
    assert result.termination_condition == TerminationCondition.optimal
    assert abs(pyo.value(m.x)-capacity/2) < 1e-7
    observations.append((capacity,pyo.value(m.x),pyo.value(m.profit)))
observations


## Failed solves must not look like solutions
Do not load a solution until termination has been checked. With `load_solution=False`, the interface can report infeasibility without trying to load nonexistent values. Never display values left over from an earlier successful solve as if they solved the new model.


In [ ]:
m.capacity.set_value(-1)
solver.config.load_solution = False
result = solver.solve(m)
assert result.termination_condition == TerminationCondition.infeasible
print('Correctly detected infeasible model; previous variable values are not a solution.')


## Your experiment
Restore a feasible capacity and explicitly re-enable solution loading. Predict the objective before re-solving. Explain why a mutable parameter is useful and when rebuilding the model would be clearer.
